# 04 · Tuning de los tres mejores candidatos

30 configuraciones por cada una de las tres mejores familias **en cada horizonte** (180 nested runs). Búsqueda aleatoria reproducible con ParameterSampler y evaluación explícita en los mismos folds; facilita registrar cada modelo y su evidencia.

In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.datos import DATA, DOCS, EVIDENCE
from src.datos import execute
HAS_DATA = (DATA / "obt.parquet").exists()
display(Markdown("**OBT real local disponible**" if HAS_DATA else
    "**PENDIENTE:** faltan credenciales Snowflake válidas y snapshot. Este notebook no contiene resultados empíricos del ECU 911."))


2026/11/11 17:32:44 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /Users/feliperodriguez/pset_3_entorno/lib/python3.14/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


**PENDIENTE:** faltan credenciales Snowflake válidas y snapshot. Este notebook no contiene resultados empíricos del ECU 911.

## Espacios predefinidos

Regularización y learning rate en escala logarítmica; profundidad, hojas y tamaños en rangos acotados. Se evalúa la mejora sobre el default. Si no supera la mayor SD entre pliegues de ambas versiones, se conserva el default para selección. No se abre test.

In [2]:
from src.datos import models, spaces
for name in models():
    print(name, spaces(name))
if HAS_DATA:
    tuned = execute("tuning")
    display(pd.DataFrame(tuned)[["horizon", "name", "cv_ap_mean", "cv_ap_std", "improvement", "improvement_exceeds_std", "params", "run_id"]])
    display(pd.read_csv(EVIDENCE / "tuning_trials.csv"))

logistica {'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x1215196a0>, 'class_weight': [None, 'balanced'], 'max_iter': [500, 1000]}
random_forest {'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x1214cd6d0>, 'max_depth': [None, 6, 12, 20], 'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x1213d7f00>, 'max_features': ['sqrt', 0.3, 0.7], 'class_weight': [None, 'balanced']}
hist_boosting {'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x1215227b0>, 'max_iter': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x121523af0>, 'max_leaf_nodes': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x1215382f0>, 'l2_regularization': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x12152c120>, 'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x12152c390>}
mlp {'hidden_layer_sizes': [(32,), 

La búsqueda de hiperparámetros y la selección usan la misma CV: puede haber optimismo de selección. El hold-out final permanece independiente y solo se abre después de congelar finalistas y umbrales.